# Merger Histories and Origin Analysis

This notebook analyzes the temporal evolution and origin of posterior associations, including:
- Cluster trajectory evolution from early times to z=0
- Constrained vs control comparisons
- Localization metrics (hull volume ratio, scatter ratio, information gain)
- Mass distributions across associations
- Individual cluster diagnostic inspection

For general association analysis, see `posterior_associations_general.ipynb`.

In [ ]:
# Cell 1: Imports and Setup
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
from collections import defaultdict

# Add parent directory to path
sys.path.insert(0, os.path.abspath('..'))

from backend.config_loader import load_config
from backend.utils import *
from backend.analysis import (
    find_control_matches_and_recenter_single,
    analyze_volume_ratios_batch,
    save_localization_metrics_to_hdf5,
    load_localization_metrics_from_hdf5,
)
from backend.math_utils import (
    _mean_matter_density_Msun_per_Mpc3,
    _dimensionless_covariance_metrics
)
from backend.trace_processing import _get_initial_final_positions

# Plotting style
try:
    from pymanticore.analysis.matplotlib import get_mplstyle_path, ManticoreColors
    mnras_style = "mnras"
    USE_MNRAS_STYLE = True
except ImportError:
    USE_MNRAS_STYLE = False
    print("pymanticore not found, using default matplotlib style")

In [ ]:
# Cell 2: Configuration Loading
config = load_config(config_path="../config.toml")

print("Configuration loaded:")
print(f"Base directory: {config.global_config.basedir}")
print(f"Output directory: {config.global_config.output_dir}")
print(f"Observer coordinates: {config.global_config.observer_coords}")
print(f"Mode1 - eps: {config.mode1.eps}, min_samples: {config.mode1.min_samples}")
print(f"Mode2 - target_snapshot: {config.mode2.target_snapshot}, min_cluster_size: {config.mode2.min_cluster_size}")

MIN_CLUSTER_SIZE = config.mode2.min_cluster_size
EPS = config.mode1.eps
MIN_SAMPLES = config.mode1.min_samples
OUTPUT_DIR = os.path.join("..", config.global_config.output_dir)

In [ ]:
# Cell 3: Load Clusters, Selection Cuts, and Check Traces
import yaml

# Load selection cuts from shared YAML file
with open('selection_cuts.yaml', 'r') as f:
    SELECTION_CUTS = yaml.safe_load(f)

# Convert YAML format [min, max] to tuple format (min, max)
for selection_name, cuts in SELECTION_CUTS.items():
    if cuts:  # Skip empty dicts (like 'Raw')
        SELECTION_CUTS[selection_name] = {k: tuple(v) for k, v in cuts.items()}

# Quality selection: "Raw", "Fiducial", or "Strict"
QUALITY_SELECTION = "Fiducial"
print(f"Quality selection: {QUALITY_SELECTION}")
print(f"Cuts: {SELECTION_CUTS[QUALITY_SELECTION]}")

# Convert eps to filename format (2.5 -> "2p5")
eps_str = str(EPS).replace('.', 'p')
filename = f"dbscan_clusters_eps_{str(EPS)}_ms_{MIN_SAMPLES}.h5"

# Load clusters for Mode 2 tracing (legacy format)
clusters, cluster_metadata = load_clusters_from_hdf5(OUTPUT_DIR, filename=filename, minimal=False)
print(f"Successfully loaded {len(clusters)} clusters")
clusters_available = len(clusters) > 0

# Also load HDBSCAN catalog to get selection cut fields (ambiguity_rate, etc.)
from backend.io import load_hdbscan_clusters_from_hdf5
hdbscan_clusters, _, _ = load_hdbscan_clusters_from_hdf5(
    OUTPUT_DIR, filename, min_existence_prob=0.0, load_members=False
)

# Build lookup dict: cluster_id -> hdbscan cluster data
hdbscan_lookup = {c['cluster_id']: c for c in hdbscan_clusters}

def apply_quality_cuts(cluster_ids, selection_name):
    """Filter cluster_ids based on quality cuts using HDBSCAN catalog data."""
    if selection_name not in SELECTION_CUTS:
        raise ValueError(f"Unknown selection: {selection_name}. Choose from {list(SELECTION_CUTS.keys())}")
    
    cuts = SELECTION_CUTS[selection_name]
    if not cuts:
        return set(cluster_ids)  # Raw - no filtering
    
    filtered_ids = set()
    for cid in cluster_ids:
        if cid not in hdbscan_lookup:
            continue
        
        c = hdbscan_lookup[cid]
        
        # Extract values for comparison
        values = {
            'ambiguity_rate': c.get('ambiguity_rate', 0),
            'n_members': c.get('n_members', 0),
            'sigma_R': np.linalg.norm(c.get('position_std', [0, 0, 0])),
            'sigma_log_M': c.get('log10_m200_mass_std', 0),
        }
        
        # Check all cuts
        passes = True
        for key, (lo, hi) in cuts.items():
            val = values.get(key, 0)
            if lo is not None and val < lo:
                passes = False
                break
            if hi is not None and val > hi:
                passes = False
                break
        
        if passes:
            filtered_ids.add(cid)
    
    return filtered_ids

# Get set of cluster_ids that pass selection cuts
all_cluster_ids = [c['cluster_id'] for c in clusters]
selected_cluster_ids = apply_quality_cuts(all_cluster_ids, QUALITY_SELECTION)
print(f"Clusters passing '{QUALITY_SELECTION}' cuts: {len(selected_cluster_ids)} / {len(all_cluster_ids)}")

# Check for traces file existence without loading data
trace_filename = f"halo_traces_eps_{eps_str}_min_samples_{MIN_SAMPLES}.h5"

try:
    trace_filepath = os.path.join(OUTPUT_DIR, trace_filename)
    if os.path.exists(trace_filepath):
        print("Halo trace file found")
        traces_available = True
    else:
        print("No halo trace data found. Run Mode 2 for temporal evolution plots.")
        traces_available = False
except Exception:
    print("No halo trace data found. Run Mode 2 for temporal evolution plots.")
    traces_available = False

In [ ]:
# Cell 4: Data Summary Statistics
if clusters_available:
    cluster_sizes = [c['cluster_size'] for c in clusters]
    
    print("Cluster Statistics (All):")
    print(f"  Total clusters: {len(clusters)}")
    print(f"  Largest cluster size: {max(cluster_sizes) if cluster_sizes else 0}")
    print(f"  Mean cluster size: {np.mean(cluster_sizes):.2f}")
    print(f"  Median cluster size: {np.median(cluster_sizes):.2f}")
    mask = np.where(np.array(cluster_sizes) >= MIN_CLUSTER_SIZE)
    print(f"  Num clusters above {MIN_CLUSTER_SIZE} = {len(mask[0])}")
    
    # Stats for selected clusters
    selected_clusters = [c for c in clusters if c['cluster_id'] in selected_cluster_ids]
    if selected_clusters:
        selected_sizes = [c['cluster_size'] for c in selected_clusters]
        print(f"\nCluster Statistics ('{QUALITY_SELECTION}' selection):")
        print(f"  Selected clusters: {len(selected_clusters)}")
        print(f"  Largest cluster size: {max(selected_sizes)}")
        print(f"  Mean cluster size: {np.mean(selected_sizes):.2f}")
        print(f"  Median cluster size: {np.median(selected_sizes):.2f}")
          
    print(f"\nTop 5 clusters by size (from selected):")
    sorted_selected = sorted(selected_clusters, key=lambda x: x['cluster_size'], reverse=True)
    for i, cluster in enumerate(sorted_selected[:5]):
        print(f"  {i+1}. Cluster {cluster['cluster_id']}: {cluster['cluster_size']} members, "
              f"mass={cluster['mean_m200_mass']:.2e}")

if traces_available:
    cluster_trace_counts = get_cluster_trace_info(OUTPUT_DIR, filename=trace_filename)
    total_traced_haloes = sum(cluster_trace_counts.values())
    
    # Count traces for selected clusters
    selected_trace_counts = {k: v for k, v in cluster_trace_counts.items() if k in selected_cluster_ids}
    total_selected_traces = sum(selected_trace_counts.values())
    
    print(f"\nTrace Statistics:")
    print(f"  Total traced haloes: {total_traced_haloes}")
    print(f"  Clusters with traces: {len(cluster_trace_counts)}")
    print(f"  Traces for '{QUALITY_SELECTION}' clusters: {total_selected_traces}")
    print(f"  Selected clusters with traces: {len(selected_trace_counts)}")

---
## Helper Functions

In [ ]:
# Cell 5: plot_cluster_trajectory()
def plot_cluster_trajectory(cluster_id, ax=None):
    """
    Plot XY projection trajectories for all halos in a cluster.
    Shows paths from early times to z=0 with start/end markers and radial reference circles.
    """
    traces = load_single_cluster_traces(cluster_id, OUTPUT_DIR, filename=trace_filename)
    
    if traces is None:
        print(f"No traces available for cluster {cluster_id}")
        return None
    
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(12, 12))
        standalone = True
    else:
        standalone = False
    
    # Calculate cluster centroid from final positions (snapshot 77)
    final_positions = []
    final_masses = []
    for trace_data in traces:
        positions = trace_data['BoundSubhalo/CentreOfMass']
        masses = trace_data['SO/200/crit/TotalMass']
        snapshots = trace_data['snapshots']
        final_idx = np.where(snapshots == 77)[0]
        if len(final_idx) > 0:
            final_positions.append(positions[final_idx[0]])
            final_masses.append(masses[final_idx[0]])
    
    if len(final_positions) > 0:
        final_positions = np.array(final_positions)
        cluster_centroid = np.mean(final_positions, axis=0)
        total_final_mass = np.mean(final_masses)
    else:
        cluster_centroid = np.array([0, 0, 0])
        total_final_mass = 0
    
    # Generate colors for each trajectory
    colors = plt.cm.tab20(np.linspace(0, 1, min(20, len(traces))))
    if len(traces) > 20:
        colors = plt.cm.gist_ncar(np.linspace(0, 1, len(traces)))
    
    for i, trace_data in enumerate(traces):
        positions = trace_data['BoundSubhalo/CentreOfMass']
        snapshots = trace_data['snapshots']
        
        # Plot trajectory line
        ax.plot(positions[:, 0], positions[:, 1], '-', 
               alpha=0.5, linewidth=0.8, color=colors[i])
        
        # Plot start and end points with smaller sizes
        ax.scatter(positions[0, 0], positions[0, 1], 
                  c='red', s=8, marker='s', alpha=0.9, 
                  edgecolors='darkred', linewidth=0.3, zorder=10)
        ax.scatter(positions[-1, 0], positions[-1, 1], 
                  c='blue', s=8, marker='o', alpha=0.9,
                  edgecolors='darkblue', linewidth=0.3, zorder=10)
    
    # Add radial ring at 10 Mpc
    circle_10 = plt.Circle((cluster_centroid[0], cluster_centroid[1]), 
                          10, fill=False, color='gray', linestyle='--', 
                          linewidth=1, alpha=0.7)
    ax.add_patch(circle_10)
    
    # Set axis limits to +/- 15 Mpc from cluster centroid
    ax.set_xlim(cluster_centroid[0] - 15, cluster_centroid[0] + 15)
    ax.set_ylim(cluster_centroid[1] - 15, cluster_centroid[1] + 15)
    
    # Add text box with cluster information in lower left corner
    info_text = f'Association {cluster_id} (n={len(traces)})\n' + r"$\langle M_{\mathrm{200}}$($z=0) \rangle$ =" + f'{total_final_mass:.1e}' + r' M$_{\odot}$'
    ax.text(0.05, 0.05, info_text, transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
            verticalalignment='bottom', zorder=10)
    
    ax.set_xlabel('X (Mpc)')
    ax.set_ylabel('Y (Mpc)')
    ax.set_aspect('equal')
    ax.set_facecolor('#f8f8f8')
    ax.tick_params(labelsize=8)
    
    if standalone:
        plt.tight_layout()
        plt.show()
    
    return traces

In [ ]:
# Cell 6: find_control_matches_and_recenter() - wrapper for local use
def find_control_matches_and_recenter(cluster_id, mass_tolerance_dex=0.1):
    """
    Find control matches for each halo in a constrained cluster and recenter them.
    Uses the backend function with local config.
    """
    return find_control_matches_and_recenter_single(
        cluster_id,
        config=config,
        trace_filename=trace_filename,
        mass_tolerance_dex=mass_tolerance_dex,
        output_dir=OUTPUT_DIR,
    )

In [ ]:
# Cell 7: plot_cluster_mass_evolution()
def plot_cluster_mass_evolution(cluster_id, ax=None):
    """
    Plot the median and 10th-90th percentile range of BoundSubhalo/TotalMass
    as a function of snapshot for all haloes in the given cluster.
    """
    traces = load_single_cluster_traces(cluster_id, OUTPUT_DIR, filename=trace_filename)
    if traces is None or len(traces) == 0:
        print(f"No traces available for cluster {cluster_id}")
        return

    # Gather all unique snapshot indices present
    all_snaps = np.unique(np.concatenate([t['snapshots'] for t in traces]))
    all_snaps.sort()

    # For each snapshot, collect the masses from each trace (where available)
    medians = []
    p10 = []
    p90 = []

    for snap in all_snaps:
        masses_at_snap = []
        for t in traces:
            idx = np.where(t['snapshots'] == snap)[0]
            if idx.size > 0:
                masses_at_snap.append(t['BoundSubhalo/TotalMass'][idx[0]])
        if len(masses_at_snap) > 0:
            arr = np.array(masses_at_snap)
            p10.append(np.percentile(arr, 10))
            medians.append(np.percentile(arr, 50))
            p90.append(np.percentile(arr, 90))
        else:
            p10.append(np.nan)
            medians.append(np.nan)
            p90.append(np.nan)

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
        standalone = True
    else:
        standalone = False

    all_snaps = np.array(all_snaps)
    p10 = np.array(p10)
    medians = np.array(medians)
    p90 = np.array(p90)

    ax.plot(all_snaps, medians, '-', lw=2, label='Median Mass', color='C0')
    ax.fill_between(all_snaps, p10, p90, color='C0', alpha=0.3,
                    label='10th-90th percentile')

    ax.set_xlabel('Snapshot', fontsize=12)
    ax.set_ylabel('BoundSubhalo / TotalMass', fontsize=12)
    ax.set_title(f'Cluster {cluster_id}: Mass Evolution (n={len(traces)})', pad=10)
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=10)
    ax.tick_params(labelsize=10)

    if standalone:
        plt.tight_layout()
        plt.show()

In [ ]:
# Cell 8: plot_cluster_mass_distribution()
def plot_cluster_mass_distribution(cluster_id, ax=None):
    """
    Plot histogram of M200 masses for a single cluster.
    """
    if not clusters_available:
        print(f"No clusters available")
        return
    
    # Find the cluster
    target_cluster = None
    for cluster in clusters:
        if cluster['cluster_id'] == cluster_id:
            target_cluster = cluster
            break
    
    if target_cluster is None:
        print(f"Cluster ID {cluster_id} not found")
        return
    
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
        standalone = True
    else:
        standalone = False
    
    cluster_masses = target_cluster['member_data']['BoundSubhalo/TotalMass']
    cluster_size = target_cluster['cluster_size']
    median_mass = np.median(cluster_masses)
    mean_mass = target_cluster['mean_m200_mass']
    
    ax.hist(cluster_masses, bins=10**np.arange(13., 16, 0.05), alpha=0.7, 
            color='skyblue', edgecolor='black', linewidth=0.5)
    
    ax.axvline(mean_mass, color='red', linestyle='--', linewidth=2, alpha=0.8, 
               label=f'Mean: {mean_mass:.2e}')
    ax.axvline(median_mass, color='orange', linestyle='-', linewidth=2, alpha=0.8, 
               label=f'Median: {median_mass:.2e}')
    
    ax.set_xlabel(r'M200 Mass (M$_\odot$)', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_title(f'Cluster {cluster_id} (n={cluster_size})', fontsize=12, pad=10)
    ax.tick_params(labelsize=8)
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.set_xscale("log")
    ax.legend(fontsize=8)
    
    if standalone:
        plt.tight_layout()
        plt.show()

In [ ]:
# Cell 9: plot_cluster_diagnostic()
def plot_cluster_diagnostic(cluster_id=None, coordinates=None):
    """
    Plot 3-panel spatial diagnostic (XY, XZ, YZ projections) for a cluster
    or custom coordinates.
    """
    if coordinates is not None:
        target_cluster_center = np.array(coordinates)
        target_cluster_id = "Custom"
        cluster_type = "Custom"
        target_cluster = {
            'cluster_size': 'N/A',
            'mean_mass': 'N/A',
            'mass_std': 'N/A'
        }
    else:
        if not clusters_available or len(clusters) == 0:
            print("No clusters available for diagnostic plot")
            return
        
        if cluster_id is None:
            target_cluster = max(clusters, key=lambda x: x['cluster_size'])
            cluster_type = "Largest"
        else:
            target_cluster = None
            for cluster in clusters:
                if cluster['cluster_id'] == cluster_id:
                    target_cluster = cluster
                    break
            
            if target_cluster is None:
                print(f"Cluster ID {cluster_id} not found")
                return
            cluster_type = "Selected"
        
        target_cluster_center = target_cluster['mean_position']
        target_cluster_id = target_cluster['cluster_id']
    
    # Get all positions for plotting context
    if coordinates is None:
        all_positions = []
        all_cluster_ids = []
        
        for cluster in clusters:
            positions = cluster['member_data']['BoundSubhalo/CentreOfMass']
            cluster_id_val = cluster['cluster_id']
            all_positions.extend(positions)
            all_cluster_ids.extend([cluster_id_val] * len(positions))
        
        all_positions = np.array(all_positions)
        all_cluster_ids = np.array(all_cluster_ids)
        
        distances = np.linalg.norm(all_positions - target_cluster_center, axis=1)
        within_15mpc = distances <= 15.0
        
        nearby_positions = all_positions[within_15mpc]
        nearby_cluster_labels = all_cluster_ids[within_15mpc]
    else:
        if not clusters_available or len(clusters) == 0:
            nearby_positions = np.array([]).reshape(0, 3)
            nearby_cluster_labels = np.array([])
        else:
            all_positions = []
            all_cluster_ids = []
            
            for cluster in clusters:
                positions = cluster['member_data']['BoundSubhalo/CentreOfMass']
                cluster_id_val = cluster['cluster_id']
                all_positions.extend(positions)
                all_cluster_ids.extend([cluster_id_val] * len(positions))
            
            all_positions = np.array(all_positions)
            all_cluster_ids = np.array(all_cluster_ids)
            
            distances = np.linalg.norm(all_positions - target_cluster_center, axis=1)
            within_15mpc = distances <= 15.0
            
            nearby_positions = all_positions[within_15mpc]
            nearby_cluster_labels = all_cluster_ids[within_15mpc]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    if len(nearby_cluster_labels) > 0:
        unique_labels = np.unique(nearby_cluster_labels)
        colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
    else:
        unique_labels = []
        colors = []
    
    projections = [
        (0, 1, 'X', 'Y', 'X-Y'),
        (0, 2, 'X', 'Z', 'X-Z'), 
        (1, 2, 'Y', 'Z', 'Y-Z')
    ]
    
    for ax_idx, (dim1, dim2, label1, label2, proj_name) in enumerate(projections):
        ax = axes[ax_idx]
        
        for i, label in enumerate(unique_labels):
            mask = nearby_cluster_labels == label
            if coordinates is None and label == target_cluster_id:
                ax.scatter(nearby_positions[mask, dim1], nearby_positions[mask, dim2], 
                          c='gray', s=80, alpha=0.8, label=f'{cluster_type} Cluster (ID {label})', 
                          edgecolors='darkred')
            else:
                ax.scatter(nearby_positions[mask, dim1], nearby_positions[mask, dim2], 
                          c=[colors[i]], s=40, alpha=0.6, label=f'Cluster {label}')
        
        center_label = 'Custom Center' if coordinates is not None else 'Cluster Center'
        ax.scatter(target_cluster_center[dim1], target_cluster_center[dim2], 
                  c='black', s=300, marker='*', label=center_label, edgecolors='white', linewidth=2)
        
        circle = plt.Circle((target_cluster_center[dim1], target_cluster_center[dim2]), 
                           7.5, fill=False, color='black', linestyle='--', linewidth=2, 
                           label='7.5 Mpc (eps threshold)')
        ax.add_patch(circle)
        
        ax.set_xlabel(f'{label1} (Mpc)')
        ax.set_ylabel(f'{label2} (Mpc)')
        ax.set_title(f'{proj_name} projection')
        ax.set_aspect('equal')
        
        if ax_idx == 0:
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    if coordinates is not None:
        fig.suptitle(f'Haloes within 15 Mpc of Custom Coordinates\n'
                    f'Center: [{target_cluster_center[0]:.1f}, {target_cluster_center[1]:.1f}, {target_cluster_center[2]:.1f}]', 
                    fontsize=14)
        print(f"Custom coordinates analysis:")
        print(f"  Center: [{target_cluster_center[0]:.1f}, {target_cluster_center[1]:.1f}, {target_cluster_center[2]:.1f}]")
    else:
        fig.suptitle(f'Haloes within 15 Mpc of {cluster_type} Cluster\n'
                    f'Cluster Size: {target_cluster["cluster_size"]}, ID: {target_cluster_id}', 
                    fontsize=14)
        print(f"{cluster_type} cluster analysis:")
        print(f"  Cluster ID: {target_cluster_id}")
        print(f"  Size: {target_cluster['cluster_size']} members")
        print(f"  Center: [{target_cluster_center[0]:.1f}, {target_cluster_center[1]:.1f}, {target_cluster_center[2]:.1f}]")
        print(f"  Mean m200 mass: {target_cluster['mean_m200_mass']:.2e}")
        print(f"  Mass m200 std: {target_cluster['m200_mass_std']:.2e}")
    
    plt.tight_layout()
    plt.show()

---
## Helper Functions

**Note:** The trajectory evolution and constrained vs control plots use `QUALITY_SELECTION` (set in Cell 3) to filter clusters. Change it to `"Raw"`, `"Fiducial"`, or `"Strict"` to select different quality cuts. The localization metrics plot (Cell 16) automatically compares all three selections.

In [ ]:
# Cell 10: Temporal Evolution Setup (filtered by selection cuts)
if traces_available:
    cluster_trace_counts = get_cluster_trace_info(OUTPUT_DIR, filename=trace_filename)
    
    # Filter clusters with sufficient traces AND passing selection cuts
    min_traces_for_plot = 5
    significant_trace_clusters = {
        k: v for k, v in cluster_trace_counts.items() 
        if v >= min_traces_for_plot and k in selected_cluster_ids
    }
    
    print(f"Temporal evolution setup ('{QUALITY_SELECTION}' selection):")
    print(f"  Total clusters with traces: {len(cluster_trace_counts)}")
    print(f"  Clusters passing selection cuts: {len([k for k in cluster_trace_counts if k in selected_cluster_ids])}")
    print(f"  Clusters with >= {min_traces_for_plot} traces AND passing cuts: {len(significant_trace_clusters)}")
    
    # Sort by number of traces
    sorted_trace_clusters = sorted(significant_trace_clusters.items(), 
                                  key=lambda x: x[1], reverse=True)
    
    print(f"\nTop clusters by trace count (passing '{QUALITY_SELECTION}' cuts):")
    for cluster_id, trace_count in sorted_trace_clusters[:10]:
        print(f"  Cluster {cluster_id}: {trace_count} traces")
else:
    sorted_trace_clusters = []

In [ ]:
# Cell 11: Top 9 Cluster Trajectory Grid (3x3)
if traces_available and len(sorted_trace_clusters) > 0:
    # Get style context
    if USE_MNRAS_STYLE:
        style_context = plt.style.context(get_mplstyle_path(mnras_style))
    else:
        from contextlib import nullcontext
        style_context = nullcontext()
    
    with style_context:
        n_clusters_to_plot = min(9, len(sorted_trace_clusters))
        fig, axes = plt.subplots(3, 3, figsize=(6.5, 6.5))
        axes = axes.flatten()
        
        for plot_idx in range(n_clusters_to_plot):
            cluster_id, trace_count = sorted_trace_clusters[plot_idx]
            ax = axes[plot_idx]
            plot_cluster_trajectory(cluster_id, ax=ax)
        
        # Hide unused subplots
        for plot_idx in range(n_clusters_to_plot, 9):
            axes[plot_idx].set_visible(False)

        # Hide axis labels for inner plots
        for plot_idx in range(n_clusters_to_plot):
            ax = axes[plot_idx]
            if plot_idx not in [6, 7, 8]:
                ax.set_xlabel("")
            if plot_idx not in [0, 3, 6]:
                ax.set_ylabel("")
                
        plt.tight_layout(pad=0.1)
        os.makedirs("./plots", exist_ok=True)
        plt.savefig("./plots/cluster_evolutions.pdf")
        plt.show()
else:
    print("No traces available for trajectory plots. Run Mode 2 first.")

---
## Constrained vs Control Comparison

In [ ]:
# Cell 12: plot_constrained_vs_control_comparison()
def plot_constrained_vs_control_comparison(
    cluster_ids,
    mass_tolerance_dex=0.1,
    target_snapshot=10,
):
    """
    2x2 comparison for exactly two cluster IDs:
      Left  : constrained trajectories (XY)
      Right : controls (1:1 translated to match final positions)
    """
    from scipy.spatial import ConvexHull

    def _lagrangian_radius_from_mass(M200, rho_m):
        return (3.0 * M200 / (4.0 * np.pi * rho_m)) ** (1.0/3.0) if M200 is not None else None

    if len(cluster_ids) != 2:
        print("Function requires exactly 2 cluster IDs")
        return

    rho_m = _mean_matter_density_Msun_per_Mpc3()
    
    # Get style context
    if USE_MNRAS_STYLE:
        style_context = plt.style.context(get_mplstyle_path(mnras_style))
    else:
        from contextlib import nullcontext
        style_context = nullcontext()
    
    with style_context:
        fig, axes = plt.subplots(2, 2, figsize=(3.75, 3.75))
        final_snap = 77
        init_snap = target_snapshot

        for row_idx, cluster_id in enumerate(cluster_ids):
            # Load constrained + controls
            constrained_traces, control_traces = find_control_matches_and_recenter_single(
                cluster_id,
                config=config,
                trace_filename=trace_filename,
                mass_tolerance_dex=mass_tolerance_dex,
                output_dir=OUTPUT_DIR,
            )
            
            if not constrained_traces or not control_traces:
                print(f"Skipping cluster {cluster_id}: no matches found")
                continue

            # Get initial and final positions for metrics
            Xinit_data, Xfin_data = _get_initial_final_positions(constrained_traces, init_snap, final_snap)
            Xinit_ctrl, Xfin_ctrl = _get_initial_final_positions(control_traces, init_snap, final_snap)

            # Calculate metrics
            m_final = []
            for tr in constrained_traces:
                snaps = tr['snapshots']
                ff = np.where(snaps == final_snap)[0]
                if len(ff) == 0:
                    continue
                for mkey in ('BoundSubhalo/TotalMass', 'SO/200_crit/TotalMass'):
                    if mkey in tr:
                        m_final.append(tr[mkey][ff[0]])
                        break
            
            M200_mean = float(np.mean(m_final)) if m_final else None
            R_L = _lagrangian_radius_from_mass(M200_mean, rho_m)

            if Xinit_data.shape[0] >= 2 and Xinit_ctrl.shape[0] >= 2 and R_L is not None:
                metrics = _dimensionless_covariance_metrics(Xinit_data, Xinit_ctrl, R_L)
                s_ratio = metrics["s_ratio"]
                info_bits = metrics["info_bits"]
            else:
                s_ratio = np.nan
                info_bits = np.nan

            # Convex hull volumes
            def _hull_vol(P):
                if P.shape[0] < 4:
                    return np.nan
                try:
                    return ConvexHull(P).volume
                except Exception:
                    return np.nan

            vol_data = _hull_vol(Xinit_data)
            vol_ctrl = _hull_vol(Xinit_ctrl)
            vol_ratio = (vol_ctrl / vol_data) if (np.isfinite(vol_ctrl) and np.isfinite(vol_data) and vol_data > 0) else np.nan

            print(f"Cluster {cluster_id} metrics:")
            print(f"  Hull volume ratio: {vol_ratio:.2f}x" if not np.isnan(vol_ratio) else "  Hull volume ratio: nan")
            print(f"  s_ratio: {s_ratio:.2f}" if not np.isnan(s_ratio) else "  s_ratio: nan")
            print(f"  Info bits: {info_bits:.2f}" if not np.isnan(info_bits) else "  Info bits: nan")

            # Get cluster centroid for framing
            finals = []
            for tr in constrained_traces:
                snaps = tr['snapshots']
                ff = np.where(snaps == final_snap)[0]
                if len(ff) == 0:
                    continue
                for pkey in ('BoundSubhalo/CentreOfMass', 'SO/200_crit/CentreOfMass'):
                    if pkey in tr:
                        finals.append(tr[pkey][ff[0]])
                        break
            finals = np.asarray(finals)
            cluster_centroid = finals.mean(axis=0) if finals.size else np.zeros(3)

            # Left panel: constrained trajectories
            axL = axes[row_idx, 0]
            axL.set_facecolor('#f8f8f8')
            colors = plt.cm.viridis(np.linspace(0, 1, max(2, len(constrained_traces))))
            for i, tr in enumerate(constrained_traces):
                poskey = 'BoundSubhalo/CentreOfMass' if 'BoundSubhalo/CentreOfMass' in tr else 'SO/200_crit/CentreOfMass'
                pos = tr[poskey]
                axL.plot(pos[:, 0], pos[:, 1], '-', lw=0.5, alpha=0.6, color=colors[i % len(colors)])
                axL.scatter(pos[0, 0], pos[0, 1], s=8, c='red', alpha=0.9, marker='s',
                           edgecolors='darkred', linewidth=0.3, zorder=10)
                axL.scatter(pos[-1, 0], pos[-1, 1], s=8, alpha=0.9, c='blue', marker='o',
                           edgecolors='darkblue', linewidth=0.3, zorder=10)

            for r, ls in [(10, '--'), (20, ':')]:
                axL.add_patch(plt.Circle((cluster_centroid[0], cluster_centroid[1]), r, 
                                         fill=False, ec='0.6', ls=ls, lw=1))

            axL.set_xlim(cluster_centroid[0] - 15, cluster_centroid[0] + 15)
            axL.set_ylim(cluster_centroid[1] - 15, cluster_centroid[1] + 15)
            axL.set_aspect('equal', adjustable='box')
            
            if row_idx == 1:
                axL.set_xlabel('X (Mpc)')
            axL.set_ylabel('Y (Mpc)')
            
            if M200_mean is not None:
                info_text = f'Association {cluster_id} (n={len(constrained_traces)})\n' + \
                            r"$\langle M_{\mathrm{200}}$($z=0) \rangle$ =" + f'{M200_mean:.1e}' + r' M$_{\odot}$'
            else:
                info_text = f'Association {cluster_id} (n={len(constrained_traces)})'
            
            axL.text(0.05, 0.05, info_text, transform=axL.transAxes, fontsize=8,
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
                     verticalalignment='bottom', zorder=10)

            # Right panel: control trajectories
            axR = axes[row_idx, 1]
            axR.set_facecolor('#f8f8f8')
            ctrl_colors = plt.cm.plasma(np.linspace(0, 1, max(2, len(control_traces))))
            for i, tr in enumerate(control_traces):
                poskey = 'BoundSubhalo/CentreOfMass' if 'BoundSubhalo/CentreOfMass' in tr else 'SO/200_crit/CentreOfMass'
                pos = tr[poskey]
                axR.plot(pos[:, 0], pos[:, 1], '-', lw=0.5, alpha=0.6, color=ctrl_colors[i % len(ctrl_colors)])
                axR.scatter(pos[0, 0], pos[0, 1], s=8, c='red', alpha=0.9, marker='s',
                           edgecolors='darkred', linewidth=0.3, zorder=10)
                axR.scatter(pos[-1, 0], pos[-1, 1], s=8, alpha=0.9, c='blue', marker='o',
                           edgecolors='darkblue', linewidth=0.3, zorder=10)

            for r, ls in [(10, '--'), (20, ':')]:
                axR.add_patch(plt.Circle((cluster_centroid[0], cluster_centroid[1]), r, 
                                         fill=False, ec='0.6', ls=ls, lw=1))

            axR.set_xlim(cluster_centroid[0] - 15, cluster_centroid[0] + 15)
            axR.set_ylim(cluster_centroid[1] - 15, cluster_centroid[1] + 15)
            axR.set_aspect('equal', adjustable='box')
            
            if row_idx == 1:
                axR.set_xlabel('X (Mpc)')
            
            axR.text(0.05, 0.05, f'Control (n={len(control_traces)})',
                     transform=axR.transAxes, fontsize=8,
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
                     verticalalignment='bottom', zorder=10)

        os.makedirs("./plots", exist_ok=True)
        plt.savefig("./plots/control_trace_example.pdf")
        plt.tight_layout(pad=0.1)
        plt.show()

In [ ]:
# Cell 13: Example constrained vs control comparison
if traces_available and len(sorted_trace_clusters) >= 2:
    # Select two clusters with many traces
    example_cluster_ids = [sorted_trace_clusters[0][0], sorted_trace_clusters[1][0]]
    print(f"Comparing clusters: {example_cluster_ids}")
    
    plot_constrained_vs_control_comparison(
        cluster_ids=example_cluster_ids,
        mass_tolerance_dex=0.1,
        target_snapshot=10,
    )
else:
    print("Insufficient trace data for constrained vs control comparison.")

---
## Localization Metrics (Selection Comparison)

Compares localization metrics across all selection cuts (Raw, Fiducial, Strict). Metrics are computed once for all clusters, then filtered by selection for comparison. The plot shows:
- **Hull volume ratio**: Control vs constrained convex hull volumes at early times
- **Scatter ratio**: Dimensionless scatter of control vs constrained positions  
- **Information gain**: Bits of information from constraining (log ratio of covariance determinants)

**Caching**: Computed metrics are saved to the HDF5 catalog file for faster loading on subsequent runs. Set `FORCE_RECOMPUTE_METRICS = True` in Cell 16 to recompute even if cached metrics exist.

In [ ]:
# Cell 14: compute_localization_metrics_data() with caching support
def compute_localization_metrics_data(mass_tolerance_dex=0.1, target_snapshot=10, force_recompute=False):
    """
    Compute localization metrics data for plotting, with caching support.
    
    Parameters
    ----------
    mass_tolerance_dex : float
        Mass tolerance for control matching (default: 0.1)
    target_snapshot : int
        Target snapshot for initial positions (default: 10)
    force_recompute : bool
        If True, recompute even if cached metrics exist (default: False)
    
    Returns processed data ready for visualization.
    """
    # Try to load cached metrics first (unless force_recompute)
    if not force_recompute:
        cached = load_localization_metrics_from_hdf5(
            OUTPUT_DIR, filename,
            mass_tolerance_dex=mass_tolerance_dex,
            target_snapshot=target_snapshot
        )
        if cached is not None:
            (cluster_masses, legacy_volume_ratios, s_ratios, 
             info_gain_bits, cluster_ids, distances, metadata) = cached
            
            print(f"Using cached localization metrics (computed with mass_tol={metadata['mass_tolerance_dex']}, snap={metadata['target_snapshot']})")
            
            # Build return dict
            return _build_metrics_dict(cluster_masses, legacy_volume_ratios, s_ratios, 
                                       info_gain_bits, cluster_ids)
    
    # Compute fresh metrics
    print("Computing localization metrics (this may take a while)...")
    (cluster_masses,
     legacy_volume_ratios,
     s_ratios,
     info_gain_bits,
     cluster_ids,
     distances) = analyze_volume_ratios_batch(
        clusters=clusters,
        config=config,
        trace_filename=trace_filename,
        min_cluster_size=MIN_CLUSTER_SIZE,
        mass_tolerance_dex=mass_tolerance_dex,
        target_snapshot=target_snapshot,
        output_dir=OUTPUT_DIR
    )

    if cluster_masses.size == 0:
        print("No data to process")
        return None

    # Save to HDF5 for future use
    save_localization_metrics_to_hdf5(
        cluster_masses=cluster_masses,
        legacy_volume_ratios=legacy_volume_ratios,
        s_ratios=s_ratios,
        info_gain_bits=info_gain_bits,
        cluster_ids=cluster_ids,
        distances=distances,
        output_dir=OUTPUT_DIR,
        catalog_filename=filename,
        mass_tolerance_dex=mass_tolerance_dex,
        target_snapshot=target_snapshot,
    )

    return _build_metrics_dict(cluster_masses, legacy_volume_ratios, s_ratios, 
                               info_gain_bits, cluster_ids)


def _build_metrics_dict(cluster_masses, legacy_volume_ratios, s_ratios, info_gain_bits, cluster_ids):
    """Build the metrics dictionary with binned statistics."""
    # Common binning
    mass_bins = np.logspace(np.log10(cluster_masses.min()),
                            np.log10(cluster_masses.max()), 8)
    bin_indices = np.digitize(cluster_masses, mass_bins)

    def _binstats(y):
        meds, p25s, p75s, centers = [], [], [], []
        for i in range(1, len(mass_bins)):
            msk = (bin_indices == i) & np.isfinite(y)
            if np.sum(msk) >= 2:
                vals = y[msk]
                meds.append(np.median(vals))
                p25s.append(np.percentile(vals, 25))
                p75s.append(np.percentile(vals, 75))
                centers.append(np.sqrt(mass_bins[i-1] * mass_bins[i]))
        if len(centers) == 0:
            return None
        return (np.array(centers), np.array(meds), np.array(p25s), np.array(p75s))

    hull_stats = _binstats(legacy_volume_ratios)
    s_ratio_stats = _binstats(s_ratios)
    info_stats = _binstats(info_gain_bits)

    def _safe_geom_mean(x):
        x = x[np.isfinite(x) & (x > 0)]
        return np.exp(np.mean(np.log(x))) if x.size else np.nan

    print("\nLocalization Metrics Summary:")
    print(f"  N clusters: {cluster_masses.size}")
    if np.isfinite(legacy_volume_ratios).any():
        print(f"  Hull ratio (ctrl/data): median={np.nanmedian(legacy_volume_ratios):.2f}x, "
              f"geomean={_safe_geom_mean(legacy_volume_ratios):.2f}x, "
              f"range=({np.nanmin(legacy_volume_ratios):.2f}x-{np.nanmax(legacy_volume_ratios):.2f}x)")
    if np.isfinite(s_ratios).any():
        print(f"  s-ratio (ctrl/data):   median={np.nanmedian(s_ratios):.2f}, "
              f"geomean={_safe_geom_mean(s_ratios):.2f}, "
              f"range=({np.nanmin(s_ratios):.2f}-{np.nanmax(s_ratios):.2f})")
    if np.isfinite(info_gain_bits).any():
        print(f"  Info gain (bits):      median={np.nanmedian(info_gain_bits):.2f}, "
              f"mean={np.nanmean(info_gain_bits):.2f}, "
              f"range=({np.nanmin(info_gain_bits):.2f}-{np.nanmax(info_gain_bits):.2f})")

    return {
        'cluster_masses': cluster_masses,
        'legacy_volume_ratios': legacy_volume_ratios,
        's_ratios': s_ratios,
        'info_gain_bits': info_gain_bits,
        'cluster_ids': np.array(cluster_ids),  # Ensure it's an array for masking
        'hull_stats': hull_stats,
        's_ratio_stats': s_ratio_stats,
        'info_stats': info_stats
    }

In [ ]:
# Cell 15: plot_localization_metrics_comparison()
def plot_localization_metrics_comparison(data, selection_masks):
    """
    Plot 3-panel localization metrics comparing different selection cuts.
    
    Parameters:
    -----------
    data : dict
        Full localization data (computed once for all clusters)
    selection_masks : dict
        Dictionary mapping selection name -> boolean mask for cluster_ids
    """
    if data is None:
        print("No data to plot")
        return

    cluster_masses = data['cluster_masses']
    legacy_volume_ratios = data['legacy_volume_ratios']
    s_ratios = data['s_ratios']
    info_gain_bits = data['info_gain_bits']

    # Colors for each selection (Raw is most permissive, Strict is most restrictive)
    selection_colors = {
        'Raw': 'gray',
        'Fiducial': 'blue', 
        'Strict': 'red'
    }
    selection_alphas = {
        'Raw': 0.3,
        'Fiducial': 0.5,
        'Strict': 0.7
    }

    # Get style context
    if USE_MNRAS_STYLE:
        style_context = plt.style.context(get_mplstyle_path(mnras_style))
    else:
        from contextlib import nullcontext
        style_context = nullcontext()

    with style_context:
        fig, axes = plt.subplots(1, 3, figsize=(6.5, 2.5), constrained_layout=True)

        # Helper to compute binned statistics for a subset
        def compute_binstats(masses, values, n_bins=8):
            if len(masses) < 2:
                return None
            valid = np.isfinite(masses) & np.isfinite(values)
            if np.sum(valid) < 2:
                return None
            masses_v = masses[valid]
            values_v = values[valid]
            
            mass_bins = np.logspace(np.log10(masses_v.min()), np.log10(masses_v.max()), n_bins)
            bin_indices = np.digitize(masses_v, mass_bins)
            
            meds, p25s, p75s, centers = [], [], [], []
            for i in range(1, len(mass_bins)):
                msk = bin_indices == i
                if np.sum(msk) >= 2:
                    vals = values_v[msk]
                    meds.append(np.median(vals))
                    p25s.append(np.percentile(vals, 25))
                    p75s.append(np.percentile(vals, 75))
                    centers.append(np.sqrt(mass_bins[i-1] * mass_bins[i]))
            if len(centers) == 0:
                return None
            return (np.array(centers), np.array(meds), np.array(p25s), np.array(p75s))

        # Plot order: Raw first (background), then Fiducial, then Strict (foreground)
        plot_order = ['Raw', 'Fiducial', 'Strict']
        
        for panel_idx, (y_data, ylabel, ref_line) in enumerate([
            (legacy_volume_ratios, 'Hull volume ratio\n(Control / Constrained)', 1.0),
            (s_ratios, 'Dimensionless scatter ratio\n$s_{\\rm ctrl}/s_{\\rm data}$', 1.0),
            (info_gain_bits, 'Information gain (bits)', 0.0)
        ]):
            ax = axes[panel_idx]
            
            for selection_name in plot_order:
                if selection_name not in selection_masks:
                    continue
                    
                mask = selection_masks[selection_name]
                if not np.any(mask):
                    continue
                
                color = selection_colors.get(selection_name, 'gray')
                alpha = selection_alphas.get(selection_name, 0.5)
                n_clusters = np.sum(mask)
                
                # Scatter points
                ax.scatter(cluster_masses[mask], y_data[mask], 
                          alpha=alpha, s=8, color=color, edgecolors='none',
                          label=f'{selection_name} (N={n_clusters})', zorder=plot_order.index(selection_name)+1)
                
                # Binned statistics (only for Fiducial and Strict to avoid clutter)
                if selection_name in ['Fiducial', 'Strict']:
                    stats = compute_binstats(cluster_masses[mask], y_data[mask])
                    if stats is not None:
                        xc, med, p25, p75 = stats
                        ax.plot(xc, med, '-o', color=color, linewidth=1.5, markersize=3, 
                               zorder=plot_order.index(selection_name)+10)
                        ax.fill_between(xc, p25, p75, color=color, alpha=0.15, zorder=plot_order.index(selection_name)+5)
            
            ax.set_xscale('log')
            if panel_idx < 2:  # Hull ratio and s_ratio use log scale
                ax.set_yscale('log')
            ax.set_xlabel(r'$\langle M_{200}(z=0) \rangle$ [M$_{\odot}$]')
            ax.set_ylabel(ylabel)
            ax.axhline(ref_line, color='k', linestyle='--', alpha=0.5, zorder=0)
            
            if panel_idx == 0:
                ax.legend(loc='upper left', fontsize=7, framealpha=0.9)

        plt.tight_layout(pad=0.1)
        os.makedirs("./plots", exist_ok=True)
        plt.savefig("./plots/trace_metric_compare.pdf")
        plt.show()

# Keep the old function for backward compatibility
def plot_localization_metrics(data):
    """Plot single-selection localization metrics (legacy interface)."""
    if data is None:
        print("No data to plot")
        return
    # Create a mask that includes all data
    all_mask = np.ones(len(data['cluster_masses']), dtype=bool)
    plot_localization_metrics_comparison(data, {'All': all_mask})

In [ ]:
# Cell 16: Run metrics analysis - compute once, compare all selection cuts
# Set to True to force recomputation even if cached metrics exist
FORCE_RECOMPUTE_METRICS = False

if traces_available and clusters_available:
    print("Loading/computing localization metrics for ALL clusters...")
    print(f"  force_recompute = {FORCE_RECOMPUTE_METRICS}")
    
    # Compute metrics for all clusters (no filtering - Raw selection)
    # Will use cached metrics if available (unless FORCE_RECOMPUTE_METRICS=True)
    localization_data = compute_localization_metrics_data(
        mass_tolerance_dex=0.1,
        target_snapshot=10,
        force_recompute=FORCE_RECOMPUTE_METRICS,
    )
    
    if localization_data is not None:
        # Get cluster_ids from the computed data
        computed_cluster_ids = localization_data['cluster_ids']
        
        # Build selection masks based on which clusters pass each cut
        # Since selections are nested (Strict ⊂ Fiducial ⊂ Raw), we compute each mask
        selection_masks = {}
        
        for selection_name in ['Raw', 'Fiducial', 'Strict']:
            # Get cluster_ids that pass this selection
            passing_ids = apply_quality_cuts(computed_cluster_ids, selection_name)
            # Create boolean mask for the computed data
            mask = np.isin(computed_cluster_ids, list(passing_ids))
            selection_masks[selection_name] = mask
            print(f"  {selection_name}: {np.sum(mask)} clusters with computed metrics")
        
        # Plot comparison of all selections
        print("\nPlotting selection comparison...")
        plot_localization_metrics_comparison(localization_data, selection_masks)
    else:
        print("No localization data computed")
else:
    print("Traces or clusters not available for localization metrics.")

---
## Mass Distributions

In [ ]:
# Cell 17: Mass Distributions Grid (5xN) - filtered by selection cuts
if clusters_available and len(clusters) > 0:
    # Set cluster size threshold
    cluster_size_threshold = MIN_CLUSTER_SIZE
    
    # Filter clusters by size AND selection cuts
    large_clusters = [
        cluster for cluster in clusters 
        if cluster['cluster_size'] >= cluster_size_threshold 
        and cluster['cluster_id'] in selected_cluster_ids
    ]
    
    if len(large_clusters) == 0:
        print(f"No clusters found with size >= {cluster_size_threshold} passing '{QUALITY_SELECTION}' cuts")
    else:
        print(f"Plotting mass distributions for {len(large_clusters)} clusters")
        print(f"  (size >= {cluster_size_threshold}, passing '{QUALITY_SELECTION}' cuts)")
        
        # Calculate grid dimensions (5 columns, as many rows as needed)
        n_cols = 5
        n_rows = (len(large_clusters) + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows))
        
        if n_rows == 1:
            axes = axes.reshape(1, -1)
        axes = axes.flatten()
        
        # Sort by cluster size descending
        large_clusters.sort(key=lambda c: c['cluster_size'], reverse=True)
        
        for plot_idx in range(len(large_clusters)):
            cluster = large_clusters[plot_idx]
            ax = axes[plot_idx]
            plot_cluster_mass_distribution(cluster['cluster_id'], ax=ax)
        
        # Hide unused subplots
        for plot_idx in range(len(large_clusters), n_rows * n_cols):
            axes[plot_idx].set_visible(False)
        
        plt.suptitle(f"Mass Distributions for Clusters with Size >= {cluster_size_threshold}\n"
                     f"({len(large_clusters)} clusters, '{QUALITY_SELECTION}' selection)", fontsize=18, y=0.95)
        plt.tight_layout()
        plt.show()
else:
    print("No clusters available for mass distribution plots.")

---
## Individual Cluster Inspection

Use the cells below to inspect individual clusters. Modify the `cluster_id` variable to explore different associations.

In [ ]:
# Cell 18: Individual cluster inspection
# Change this cluster_id to inspect different clusters
if clusters_available and traces_available:
    # Use the largest cluster by default
    cluster_id = sorted_trace_clusters[0][0] if sorted_trace_clusters else clusters[0]['cluster_id']
    
    print(f"Inspecting cluster {cluster_id}")
    print("="*50)
    
    # Spatial diagnostic
    plot_cluster_diagnostic(cluster_id=cluster_id)
    
    # Mass distribution
    plot_cluster_mass_distribution(cluster_id=cluster_id)
    
    # Trajectory plot
    _ = plot_cluster_trajectory(cluster_id=cluster_id)
    
    # Mass evolution
    plot_cluster_mass_evolution(cluster_id=cluster_id)
else:
    print("Clusters or traces not available for individual inspection.")

In [ ]:
# Cell 19: Inspect specific coordinates (optional)
# Uncomment and modify coordinates to inspect specific regions

# plot_cluster_diagnostic(coordinates=[460, 482, 470])
# plot_cluster_diagnostic(coordinates=[550, 522, 540])